In [1]:
import os
import pandas as pd
from pathlib import Path 
import re
import shutil
import subprocess
import tempfile
from contextlib import contextmanager
import numpy as np
import pandas as pd
from IPython.display import display


os.chdir("/Users/annaklimova/Hussein/")
print(f"Working dir: {os.getcwd()}")

Working dir: /Users/annaklimova/Hussein


In [2]:
def makes_txt_file(input_folder, sample_name):
    file = pd.read_csv(
        f"{input_folder}/{sample_name}/{sample_name}_MT_WT_windows.tsv",
        sep='\t', dtype=str,
    )

    frames = []
    for sequence_type, col in [("WT", "WT_peptide"), ("MT", "MUT_peptide")]:
        df = pd.DataFrame({
            "sample": sample_name,
            "sequence_type": sequence_type,
            "peptide": file[col],
            "allele": file["Allele"],
        })
        df["length"] = df["peptide"].str.len()
        frames.append(df)

    result = pd.concat(frames, ignore_index=True)

    for sequence_type, sub in result.groupby("sequence_type"):
        txt_path = f"{input_folder}/{sample_name}/{sample_name}_{sequence_type}_affinity_prediction.input.txt"
        with open(txt_path, "w") as f:
            f.write("\n".join(sub["peptide"]) + "\n")

        csv_path = f"{input_folder}/{sample_name}/{sample_name}_{sequence_type}_allele_peptide.csv"
        sub[["allele", "peptide"]].to_csv(csv_path, index=False)

    return result
 

In [3]:
OUTPUT_DIR = Path("/Users/annaklimova/Hussein/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
MIXMHCPRED_EXE = "/Users/annaklimova/Downloads/MixMHCpred/MixMHCpred"
NETMHCPAN_EXE = "/Users/annaklimova/Downloads/netMHCpan-4.2/netMHCpan"
MIXMHC2PRED_EXE = "/Users/annaklimova/Downloads/MixMHC2pred-2/MixMHC2pred"
NETMHCIIPAN_EXE = "/Users/annaklimova/Downloads/netMHCIIpan-4.3/netMHCIIpan"
MHCFLURRY_EXE = "/Users/annaklimova/miniconda3/envs/mhc-tools/bin/mhcflurry-predict"

CLASS_I_ROUTES = {
    8: {'mhcflurry': 'H-2-Kb', 'mixmhcpred': 'H2-Kb', 'netmhcpan': 'H-2-Kb'},
    9: {'mhcflurry': 'H-2-Db', 'mixmhcpred': 'H2-Db', 'netmhcpan': 'H-2-Db'},
}

CLASS_II_ROUTES = {
    15: {
        'mixmhc2pred': 'H2_Aa_b__H2_Ab_b',
        'netmhciipan': 'H-2-IAb',
        'hmmhc': 'H2-IAb',
    },
}

In [4]:
sample_names = ['ID8', 'ID8_CPR']
input_folder = "/Users/annaklimova/Hussein/mutation_peptides/"
prediction_input = pd.concat(
    [makes_txt_file(input_folder, sample) for sample in sample_names],
    ignore_index=True,
    )

# если нужен только фильтр по длинам, как раньше
prediction_input = prediction_input.loc[
    prediction_input['length'].isin([8, 9, 15])
].reset_index(drop=True)

counts = (prediction_input
          .groupby(['sample', 'sequence_type', 'length'])
          .size()
          .rename('unique_peptides')
          .reset_index())
display(counts)

,sample,sequence_type,length,unique_peptides
0,ID8,MT,8,8717
1,ID8,MT,9,9781
2,ID8,MT,15,16056
3,ID8,WT,8,8717
4,ID8,WT,9,9781
5,ID8,WT,15,16056
6,ID8_CPR,MT,8,8406
7,ID8_CPR,MT,9,9426
8,ID8_CPR,MT,15,15383
9,ID8_CPR,WT,8,8406


In [5]:
def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '-', str(value)).strip('-')
 
 
def resolve_executable(executable):
    executable = str(executable)
    if '/' in executable:
        path = Path(executable)
        if not path.is_file():
            raise FileNotFoundError(f'Executable not found: {path}')
        return str(path)
    resolved = shutil.which(executable)
    if resolved is None:
        raise FileNotFoundError(f'Executable not found on PATH: {executable}')
    return resolved
 
 
@contextmanager
def temporary_peptide_txt(peptide_series):
    path = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as handle:
            path = Path(handle.name)
            for peptide in peptide_series:
                handle.write(f'{peptide}\n')
        yield path
    finally:
        if path is not None:
            path.unlink(missing_ok=True)
            
@contextmanager
def temporary_peptide_csv(peptide_series):
    path = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as handle:
            path = Path(handle.name)
            handle.write('peptide\n')
            for peptide in peptide_series:
                handle.write(f'{peptide}\n')
        yield path
    finally:
        if path is not None:
            path.unlink(missing_ok=True)

@contextmanager
def temporary_allele_peptide_csv(allele, peptide_series):
    path = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='') as handle:
            path = Path(handle.name)
            pd.DataFrame({
                'allele': allele,
                'peptide': peptide_series,
            }).to_csv(handle, index=False)
        yield path
    finally:
        if path is not None:
            path.unlink(missing_ok=True)

In [6]:
def iter_batches(length):
    selected = prediction_input.loc[prediction_input['length'].eq(length)]
    for (sample, sequence_type), batch in selected.groupby(['sample', 'sequence_type'], sort=True):
        yield sample, sequence_type, batch.reset_index(drop=True)
 
 
def check_output(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Expected output was not created or is empty: {path}')
    print(f'Created: {path} ({path.stat().st_size:,} bytes)')
 
def run_command(command, cwd, log_path=None):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
 
    if log_path is not None:
        with open(log_path, 'w') as log_handle:
            log_handle.write(result.stdout)
            log_handle.write(result.stderr)
 
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(command)}\n"
            f"--- stdout ---\n{result.stdout}\n"
            f"--- stderr ---\n{result.stderr}"
        )
 
    return result
def show_outputs(predictor_name):
    files = sorted((OUTPUT_DIR / predictor_name).rglob('*'))
    rows = [{'file': str(p.relative_to(OUTPUT_DIR)), 'bytes': p.stat().st_size}
            for p in files if p.is_file()]
    display(pd.DataFrame(rows) if rows else pd.DataFrame(columns=['file', 'bytes']))
 

In [7]:
def run_mhcflurry():
    executable = resolve_executable(MHCFLURRY_EXE)
    predictor_dir = OUTPUT_DIR / 'mhcflurry_class_I'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}

    for length, alleles in CLASS_I_ROUTES.items():
        allele = alleles['mhcflurry']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'mhcflurry-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.output.csv'
            log = predictor_dir / f'{stem}.log'
            print(f'Running MHCflurry: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')
            with temporary_allele_peptide_csv(allele, batch['peptide']) as input_csv:
                command = [
                    executable, str(input_csv),
                    '--out', str(output),
                    '--affinity-only',
                    '--no-flanking',
                    '--backend', 'cpu',
                    '--no-throw',
                ]
                run_command(command, cwd=str(predictor_dir), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output

    return outputs

In [8]:
def run_mixmhcpred():
    executable = resolve_executable(MIXMHCPRED_EXE)
    predictor_dir = OUTPUT_DIR / 'mixmhcpred_class_I'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}
 
    for length, alleles in CLASS_I_ROUTES.items():
        allele = alleles['mixmhcpred']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'mixmhcpred-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.output.txt'
            log = predictor_dir / f'{stem}.log'
            print(f'Running MixMHCpred: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')
            with temporary_peptide_txt(batch['peptide']) as input_txt:
                command = [executable, '-i', str(input_txt), '-o', str(output), '-a', allele]
                run_command(command, cwd=str(Path(executable).parent), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output
 
    return outputs

In [9]:
def run_netmhcpan():
    executable = resolve_executable(NETMHCPAN_EXE)
    predictor_dir = OUTPUT_DIR / 'netmhcpan_class_I'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}
 
    for length, alleles in CLASS_I_ROUTES.items():
        allele = alleles['netmhcpan']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'netmhcpan-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.EL-BA.tsv'
            log = predictor_dir / f'{stem}.log'
            print(f'Running NetMHCpan: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')
            with temporary_peptide_txt(batch['peptide']) as input_txt:
                command = [
                    executable, '-p', '-f', str(input_txt), '-a', allele,
                    '-l', str(length), '-BA', '-xls', '-xlsfile', str(output),
                ]
                run_command(command, cwd=str(Path(executable).parent), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output
 
    return outputs

In [10]:
def run_mixmhc2pred():
    executable = resolve_executable(MIXMHC2PRED_EXE)
    predictor_dir = OUTPUT_DIR / 'mixmhc2pred_class_II'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}
 
    for length, alleles in CLASS_II_ROUTES.items():
        allele = alleles['mixmhc2pred']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'mixmhc2pred-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.output.txt'
            log = predictor_dir / f'{stem}.log'
            print(f'Running MixMHC2pred: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')
            with temporary_peptide_txt(batch['peptide']) as input_txt:
                command = [executable, '-i', str(input_txt), '-o', str(output), '-a', allele, '--no_context']
                run_command(command, cwd=str(Path(executable).parent), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output
 
    return outputs

In [11]:
def run_netmhciipan():
    executable = resolve_executable(NETMHCIIPAN_EXE)
    predictor_dir = OUTPUT_DIR / 'netmhciipan_class_II'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}
 
    for length, alleles in CLASS_II_ROUTES.items():
        allele = alleles['netmhciipan']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'netmhciipan-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.xls'
            log = predictor_dir / f'{stem}.log'
            print(f'Running netMHCIIpan: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')
            with temporary_peptide_txt(batch['peptide']) as input_txt:
                command = [
                    executable, '-f', str(input_txt), '-inptype', '1',
                    '-a', allele, '-BA', '-xls', '-xlsfile', str(output),
                ]
                run_command(command, cwd=str(Path(executable).parent), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output
 
    return outputs

In [12]:
HMMHC_CONDA_ENV = "hmmhc"  # окружение

def run_hmmhc():
    predictor_dir = OUTPUT_DIR / 'hmmhc_class_II'
    predictor_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}

    for length, alleles in CLASS_II_ROUTES.items():
        allele = alleles['hmmhc']
        for sample, sequence_type, batch in iter_batches(length):
            stem = f'hmmhc-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}'
            output = predictor_dir / f'{stem}.output.csv'
            log = predictor_dir / f'{stem}.log'
            print(f'Running hmMHC: {sample}, {sequence_type}, length={length}, allele={allele}, n={len(batch):,}')

            input_csv = predictor_dir / f'{stem}.input.csv'
            pd.DataFrame({'peptide': batch['peptide']}).to_csv(input_csv, index=False, header=False)
            command = [
                'conda', 'run', '-n', HMMHC_CONDA_ENV,
                'hmmhc-predict', '--allele', allele,
                '--input', str(input_csv), '--output', str(output),
            ]
            run_command(command, cwd=str(predictor_dir), log_path=log)
            check_output(output)
            outputs[(sample, sequence_type, length)] = output

    return outputs

In [13]:
 
mixmhcpred_outputs = run_mixmhcpred()
show_outputs('mixmhcpred_class_I')
 
netmhcpan_outputs = run_netmhcpan()
show_outputs('netmhcpan_class_I')

mhcflurry_outputs = run_mhcflurry()
show_outputs('mhcflurry_class_I')
 
mixmhc2pred_outputs = run_mixmhc2pred()
show_outputs('mixmhc2pred_class_II')

netmhciipan_outputs = run_netmhciipan()
show_outputs('netmhciipan_class_II')

run_hmmhc_outputs = run_hmmhc()
show_outputs('run_hmmhc_class_II')
 

Running MixMHCpred: ID8, MT, length=8, allele=H2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Kb-length-8.output.txt (472,290 bytes)
Running MixMHCpred: ID8, WT, length=8, allele=H2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Kb-length-8.output.txt (472,456 bytes)
Running MixMHCpred: ID8_CPR, MT, length=8, allele=H2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/mixmhcpred_class_I/mixmhcpred-ID8_CPR-MT-H2-Kb-length-8.output.txt (455,617 bytes)
Running MixMHCpred: ID8_CPR, WT, length=8, allele=H2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/mixmhcpred_class_I/mixmhcpred-ID8_CPR-WT-H2-Kb-length-8.output.txt (455,503 bytes)
Running MixMHCpred: ID8, MT, length=9, allele=H2-Db, n=9,781
Created: /Users/annaklimova/Hussein/predictions/mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Db-length-9.output.txt (540,283 bytes)
Running MixMHCpred: ID8, WT, length=9, allel

,file,bytes
0,mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Db-len...,130
1,mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Db-len...,540283
2,mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Kb-len...,130
3,mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Kb-len...,472290
4,mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Db-len...,130
5,mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Db-len...,540091
6,mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Kb-len...,130
7,mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Kb-len...,472456
8,mixmhcpred_class_I/mixmhcpred-ID8_CPR-MT-H2-Db...,134
9,mixmhcpred_class_I/mixmhcpred-ID8_CPR-MT-H2-Db...,520371


Running NetMHCpan: ID8, MT, length=8, allele=H-2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Kb-length-8.EL-BA.tsv (781,546 bytes)
Running NetMHCpan: ID8, WT, length=8, allele=H-2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/netmhcpan_class_I/netmhcpan-ID8-WT-H-2-Kb-length-8.EL-BA.tsv (781,701 bytes)
Running NetMHCpan: ID8_CPR, MT, length=8, allele=H-2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/netmhcpan_class_I/netmhcpan-ID8_CPR-MT-H-2-Kb-length-8.EL-BA.tsv (753,623 bytes)
Running NetMHCpan: ID8_CPR, WT, length=8, allele=H-2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/netmhcpan_class_I/netmhcpan-ID8_CPR-WT-H-2-Kb-length-8.EL-BA.tsv (753,581 bytes)
Running NetMHCpan: ID8, MT, length=9, allele=H-2-Db, n=9,781
Created: /Users/annaklimova/Hussein/predictions/netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Db-length-9.EL-BA.tsv (896,271 bytes)
Running NetMHCpan: ID8, WT, length=9, allele=H-2-Db, n

,file,bytes
0,netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Db-leng...,896271
1,netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Db-leng...,1396285
2,netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Kb-leng...,781546
3,netmhcpan_class_I/netmhcpan-ID8-MT-H-2-Kb-leng...,1244764
4,netmhcpan_class_I/netmhcpan-ID8-WT-H-2-Db-leng...,896309
5,netmhcpan_class_I/netmhcpan-ID8-WT-H-2-Db-leng...,1396152
6,netmhcpan_class_I/netmhcpan-ID8-WT-H-2-Kb-leng...,781701
7,netmhcpan_class_I/netmhcpan-ID8-WT-H-2-Kb-leng...,1244740
8,netmhcpan_class_I/netmhcpan-ID8_CPR-MT-H-2-Db-...,863635
9,netmhcpan_class_I/netmhcpan-ID8_CPR-MT-H-2-Db-...,1345977


Running MHCflurry: ID8, MT, length=8, allele=H-2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Kb-length-8.output.csv (428,440 bytes)
Running MHCflurry: ID8, WT, length=8, allele=H-2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8-WT-H-2-Kb-length-8.output.csv (428,152 bytes)
Running MHCflurry: ID8_CPR, MT, length=8, allele=H-2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8_CPR-MT-H-2-Kb-length-8.output.csv (413,647 bytes)
Running MHCflurry: ID8_CPR, WT, length=8, allele=H-2-Kb, n=8,406
Created: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8_CPR-WT-H-2-Kb-length-8.output.csv (413,327 bytes)
Running MHCflurry: ID8, MT, length=9, allele=H-2-Db, n=9,781
Created: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Db-length-9.output.csv (517,538 bytes)
Running MHCflurry: ID8, WT, length=9, allele=H-2-

,file,bytes
0,mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Db-leng...,2048
1,mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Db-leng...,517538
2,mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Kb-leng...,2048
3,mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Kb-leng...,428440
4,mhcflurry_class_I/mhcflurry-ID8-WT-H-2-Db-leng...,2048
5,mhcflurry_class_I/mhcflurry-ID8-WT-H-2-Db-leng...,517919
6,mhcflurry_class_I/mhcflurry-ID8-WT-H-2-Kb-leng...,2048
7,mhcflurry_class_I/mhcflurry-ID8-WT-H-2-Kb-leng...,428152
8,mhcflurry_class_I/mhcflurry-ID8_CPR-MT-H-2-Db-...,2052
9,mhcflurry_class_I/mhcflurry-ID8_CPR-MT-H-2-Db-...,498595


... (2 outputs left out)

Running netMHCIIpan: ID8, MT, length=15, allele=H-2-IAb, n=16,056
Created: /Users/annaklimova/Hussein/predictions/netmhciipan_class_II/netmhciipan-ID8-MT-H-2-IAb-length-15.xls (1,712,724 bytes)
Running netMHCIIpan: ID8, WT, length=15, allele=H-2-IAb, n=16,056
Created: /Users/annaklimova/Hussein/predictions/netmhciipan_class_II/netmhciipan-ID8-WT-H-2-IAb-length-15.xls (1,712,978 bytes)
Running netMHCIIpan: ID8_CPR, MT, length=15, allele=H-2-IAb, n=15,383
Created: /Users/annaklimova/Hussein/predictions/netmhciipan_class_II/netmhciipan-ID8_CPR-MT-H-2-IAb-length-15.xls (1,640,371 bytes)
Running netMHCIIpan: ID8_CPR, WT, length=15, allele=H-2-IAb, n=15,383
Created: /Users/annaklimova/Hussein/predictions/netmhciipan_class_II/netmhciipan-ID8_CPR-WT-H-2-IAb-length-15.xls (1,640,497 bytes)


,file,bytes
0,netmhciipan_class_II/netmhciipan-ID8-MT-H-2-IA...,2683358
1,netmhciipan_class_II/netmhciipan-ID8-MT-H-2-IA...,1712724
2,netmhciipan_class_II/netmhciipan-ID8-WT-H-2-IA...,2683652
3,netmhciipan_class_II/netmhciipan-ID8-WT-H-2-IA...,1712978
4,netmhciipan_class_II/netmhciipan-ID8_CPR-MT-H-...,2571217
5,netmhciipan_class_II/netmhciipan-ID8_CPR-MT-H-...,1640371
6,netmhciipan_class_II/netmhciipan-ID8_CPR-WT-H-...,2571602
7,netmhciipan_class_II/netmhciipan-ID8_CPR-WT-H-...,1640497


Running hmMHC: ID8, MT, length=15, allele=H2-IAb, n=16,056
Created: /Users/annaklimova/Hussein/predictions/hmmhc_class_II/hmmhc-ID8-MT-H2-IAb-length-15.output.csv (846,501 bytes)
Running hmMHC: ID8, WT, length=15, allele=H2-IAb, n=16,056
Created: /Users/annaklimova/Hussein/predictions/hmmhc_class_II/hmmhc-ID8-WT-H2-IAb-length-15.output.csv (846,553 bytes)
Running hmMHC: ID8_CPR, MT, length=15, allele=H2-IAb, n=15,383
Created: /Users/annaklimova/Hussein/predictions/hmmhc_class_II/hmmhc-ID8_CPR-MT-H2-IAb-length-15.output.csv (810,989 bytes)
Running hmMHC: ID8_CPR, WT, length=15, allele=H2-IAb, n=15,383
Created: /Users/annaklimova/Hussein/predictions/hmmhc_class_II/hmmhc-ID8_CPR-WT-H2-IAb-length-15.output.csv (810,845 bytes)


,file,bytes


In [27]:
def parse_mixmhcpred_file(path):
    with open(path) as fh:
        lines = [line for line in fh if not line.startswith('#')]
    df = pd.read_csv(io.StringIO(''.join(lines)), sep='\t')
    df = df.rename(columns={'Peptide': 'peptide'})
    score_col = 'Score_bestAllele' if 'Score_bestAllele' in df.columns else 'Score'
    rank_col = '%Rank_bestAllele' if '%Rank_bestAllele' in df.columns else '%Rank'
    return df[['peptide', score_col, rank_col]].rename(
        columns={score_col: 'mixmhcpred_score', rank_col: 'mixmhcpred_rank'}
    )
 
 
def parse_netmhcpan_file(path):
    df = pd.read_csv(path, sep='\t', skiprows=2)
    df = df.rename(columns={'Peptide': 'peptide'})
    keep = ['peptide']
    rename = {}
    if 'BA_score' in df.columns:
        keep.append('BA_score')
        rename['BA_score'] = 'netmhcpan_ba_score'
    rank_col = 'BA_rank' if 'BA_rank' in df.columns else 'EL_rank'
    keep.append(rank_col)
    rename[rank_col] = 'netmhcpan_rank'
    return df[keep].rename(columns=rename)

def parse_hmmhc_file(path):
    df = pd.read_csv(path)
    return df[['peptide', '-10logOdds', 'percentile_rank']].rename(
        columns={'-10logOdds': 'hmmhc_score', 'percentile_rank': 'hmmhc_rank'}
    )

def parse_mixmhc2pred_file(path):
    with open(path) as fh:
        lines = [line for line in fh if not line.startswith('#')]
    df = pd.read_csv(io.StringIO(''.join(lines)), sep='\t')
    df = df.rename(columns={'Peptide': 'peptide'})

    keep = ['peptide', '%Rank_best']
    rename = {'%Rank_best': 'mixmhc2pred_rank'}

    if 'BestAllele' in df.columns:
        keep.append('BestAllele')
        rename['BestAllele'] = 'mixmhc2pred_best_allele'

    return df[keep].rename(columns=rename)

    
def parse_mhcflurry_file(path):
    df = pd.read_csv(path)
    keep = ['peptide', 'mhcflurry_affinity']
    rename = {}
    if 'mhcflurry_affinity_percentile' in df.columns:
        keep.append('mhcflurry_affinity_percentile')
        rename['mhcflurry_affinity_percentile'] = 'mhcflurry_rank'
    return df[keep].rename(columns=rename)

def parse_netmhciipan_file(path):
    df = pd.read_csv(path, sep='\t', skiprows=2)
    df = df.rename(columns={'Peptide': 'peptide'})
    keep = ['peptide']
    rename = {}
    if 'nM' in df.columns:
        keep.append('nM')
        rename['nM'] = 'netmhciipan_affinity_nM'
    rank_col = 'Rank_BA' if 'Rank_BA' in df.columns else 'Rank_EL'
    keep.append(rank_col)
    rename[rank_col] = 'netmhciipan_rank'
    return df[keep].rename(columns=rename)

In [15]:

 def attach_predictions(results, outputs, parser):
    frames = []
    for (sample, sequence_type, length), path in outputs.items():
        parsed = parser(path)
        parsed['sample'] = sample
        parsed['sequence_type'] = sequence_type
        parsed['length'] = length
        frames.append(parsed)
    if not frames:
        return results
    combined = pd.concat(frames, ignore_index=True)
    return results.merge(combined, on=['sample', 'sequence_type', 'length', 'peptide'], how='left')
 

In [16]:
def peek_output(outputs, n=5):
    """Показать первые n строк одного случайного файла из словаря outputs"""
    sample_key, path = next(iter(outputs.items()))
    print(f'Файл: {path}')
    print(f'Ключ (sample, sequence_type, length): {sample_key}')
    print('---')
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip())

In [26]:
peek_output(run_hmmhc_outputs)
peek_output(mhcflurry_outputs)
peek_output(mixmhc2pred_outputs, n=36)
peek_output(mixmhcpred_outputs)
peek_output(netmhcpan_outputs)
peek_output(netmhciipan_outputs)


Файл: /Users/annaklimova/Hussein/predictions/hmmhc_class_II/hmmhc-ID8-MT-H2-IAb-length-15.output.csv
Ключ (sample, sequence_type, length): ('ID8', 'MT', 15)
---
peptide,-10logOdds,percentile_rank
RKASEAAVKEGKTER,27.783529984749002,8.574954237836806
KASEAAVKEGKTERT,27.388960836502505,6.199009652348052
ASEAAVKEGKTERTD,27.29123596634857,5.651034456113691
MENIFLLMKSIIHLW,33.94600907825727,92.90155237199323
Файл: /Users/annaklimova/Hussein/predictions/mhcflurry_class_I/mhcflurry-ID8-MT-H-2-Kb-length-8.output.csv
Ключ (sample, sequence_type, length): ('ID8', 'MT', 8)
---
allele,peptide,mhcflurry_affinity,mhcflurry_affinity_percentile
H-2-Kb,VKEGKTER,26718.15162928231,47.69237499999999
H-2-Kb,KEGKTERT,27917.660782590217,57.21787499999999
H-2-Kb,EGKTERTD,28309.203144694326,59.85124999999999
H-2-Kb,ENIFLLMK,21538.49430297913,23.242625000000004
Файл: /Users/annaklimova/Hussein/predictions/mixmhc2pred_class_II/mixmhc2pred-ID8-MT-H2_Aa_b__H2_Ab_b-length-15.output.txt
Ключ (sample, sequence_type, l

In [29]:
def build_and_save(length, predictor_specs, output_name):
    subset = prediction_input.loc[prediction_input['length'].eq(length)].copy()

    for outputs, parser in predictor_specs:
        subset = attach_predictions(subset, outputs, parser)

    # оставляем только те колонки-предсказания, где реально есть хоть одно не-NaN значение
    base_cols = ['sample', 'sequence_type', 'variant_id', 'peptide',
                 'counterpart_peptide', 'allele', 'length']
    base_cols = [c for c in base_cols if c in subset.columns]
    pred_cols = [c for c in subset.columns if c not in base_cols]
    pred_cols = [c for c in pred_cols if subset[c].notna().any()]

    subset = subset[base_cols + pred_cols]

    output_path = OUTPUT_DIR / output_name
    subset.to_csv(output_path, index=False)
    print(f'Saved: {output_path} ({len(subset):,} rows, columns: {list(subset.columns)})')
    display(subset)
    return subset


results_length_8 = build_and_save(
    8,
    [
        (mixmhcpred_outputs, parse_mixmhcpred_file),
        (netmhcpan_outputs, parse_netmhcpan_file),
        (mhcflurry_outputs, parse_mhcflurry_file),
    ],
    'combined_predictions_length_8.csv',
)

results_length_9 = build_and_save(
    9,
    [
        (mixmhcpred_outputs, parse_mixmhcpred_file),
        (netmhcpan_outputs, parse_netmhcpan_file),
        (mhcflurry_outputs, parse_mhcflurry_file),
    ],
    'combined_predictions_length_9.csv',
)

results_length_15 = build_and_save(
    15,
    [
        (netmhciipan_outputs, parse_netmhciipan_file),
        (mixmhc2pred_outputs, parse_mixmhc2pred_file),
        (run_hmmhc_outputs, parse_hmmhc_file),
    ],
    'combined_predictions_length_15.csv',
)

Saved: /Users/annaklimova/Hussein/predictions/combined_predictions_length_8.csv (79,870 rows, columns: ['sample', 'sequence_type', 'peptide', 'allele', 'length', 'mixmhcpred_score', 'mixmhcpred_rank', 'netmhcpan_ba_score', 'netmhcpan_rank', 'mhcflurry_affinity', 'mhcflurry_rank'])


,sample,sequence_type,peptide,allele,length,mixmhcpred_score,mixmhcpred_rank,netmhcpan_ba_score,netmhcpan_rank,mhcflurry_affinity,mhcflurry_rank
0,ID8,WT,VKEGKTEK,H2-Kb,8,-3.818722,67.314178,0.023269,84.670143,25955.459922,41.740500
1,ID8,WT,KEGKTEKT,H2-Kb,8,-3.314452,49.201007,0.011100,96.314262,27407.256137,52.231125
2,ID8,WT,EGKTEKTD,H2-Kb,8,-3.598443,59.664842,0.010246,96.597893,28371.192979,59.851250
3,ID8,WT,ENIFLLMN,H2-Kb,8,-2.614896,25.493844,0.075569,32.261105,28549.329592,62.532625
4,ID8,WT,NIFLLMNS,H2-Kb,8,-1.944639,10.352704,0.185622,7.709338,21901.257109,23.865875
...,...,...,...,...,...,...,...,...,...,...,...
79865,ID8_CPR,MT,RRFRDPEP,H2-Kb,8,-2.855330,32.847193,0.034087,69.572044,27212.463354,49.895250
79866,ID8_CPR,MT,RFRDPEPD,H2-Kb,8,-3.374656,51.420106,0.031108,73.604591,29430.603410,70.610750
79867,ID8_CPR,MT,FRDPEPDN,H2-Kb,8,-4.076315,75.372863,0.015381,94.623207,27358.286716,52.231125
79868,ID8_CPR,MT,RDPEPDNH,H2-Kb,8,-2.682179,27.425526,0.010706,96.445190,27437.404127,52.231125


Saved: /Users/annaklimova/Hussein/predictions/combined_predictions_length_9.csv (88,824 rows, columns: ['sample', 'sequence_type', 'peptide', 'allele', 'length', 'mixmhcpred_score', 'mixmhcpred_rank', 'netmhcpan_ba_score', 'netmhcpan_rank', 'mhcflurry_affinity', 'mhcflurry_rank'])


,sample,sequence_type,peptide,allele,length,mixmhcpred_score,mixmhcpred_rank,netmhcpan_ba_score,netmhcpan_rank,mhcflurry_affinity,mhcflurry_rank
0,ID8,WT,AVKEGKTEK,H2-Db,9,-2.165814,16.774625,0.013919,64.826744,19945.977559,9.816375
1,ID8,WT,VKEGKTEKT,H2-Db,9,-3.056378,43.995860,0.004879,95.477097,29689.300956,37.885125
2,ID8,WT,KEGKTEKTD,H2-Db,9,-3.942873,75.732341,0.003629,96.636246,28881.320550,33.282500
3,ID8,WT,MENIFLLMN,H2-Db,9,-3.763472,70.178971,0.017931,53.829792,29121.278205,35.475250
4,ID8,WT,ENIFLLMNS,H2-Db,9,-2.709866,31.636336,0.022232,44.559738,28808.719483,33.282500
...,...,...,...,...,...,...,...,...,...,...,...
88819,ID8_CPR,MT,RRFRDPEPD,H2-Db,9,-3.221986,50.209271,0.016849,56.536243,26695.249461,22.715875
88820,ID8_CPR,MT,RFRDPEPDN,H2-Db,9,-4.845477,90.945549,0.003829,96.451134,31005.565421,50.495250
88821,ID8_CPR,MT,FRDPEPDNH,H2-Db,9,-4.318927,85.370127,0.009040,81.850372,25283.522365,18.301750
88822,ID8_CPR,MT,RDPEPDNHY,H2-Db,9,-2.579639,27.567217,0.017846,54.025974,18954.710205,8.889625


Saved: /Users/annaklimova/Hussein/predictions/combined_predictions_length_15.csv (129,024 rows, columns: ['sample', 'sequence_type', 'peptide', 'allele', 'length', 'netmhciipan_affinity_nM', 'netmhciipan_rank', 'mixmhc2pred_rank', 'mixmhc2pred_best_allele', 'hmmhc_score', 'hmmhc_rank'])


,sample,sequence_type,peptide,allele,length,netmhciipan_affinity_nM,netmhciipan_rank,mixmhc2pred_rank,mixmhc2pred_best_allele,hmmhc_score,hmmhc_rank
0,ID8,WT,RKASEAAVKEGKTEK,I-Ab,15,15333.750000,60.105633,29.600,H2_Aa_b__H2_Ab_b,27.776613,8.334011
1,ID8,WT,KASEAAVKEGKTEKT,I-Ab,15,23308.791016,78.758347,69.400,H2_Aa_b__H2_Ab_b,27.468363,6.588230
2,ID8,WT,ASEAAVKEGKTEKTD,I-Ab,15,23588.228516,79.300674,46.800,H2_Aa_b__H2_Ab_b,27.404253,6.199010
3,ID8,WT,MENIFLLMNSIIHLW,I-Ab,15,16835.906250,64.130653,66.900,H2_Aa_b__H2_Ab_b,33.640784,91.050519
4,ID8,WT,ENIFLLMNSIIHLWK,I-Ab,15,13387.071289,54.670532,71.500,H2_Aa_b__H2_Ab_b,31.098546,56.831953
...,...,...,...,...,...,...,...,...,...,...,...
129019,ID8_CPR,MT,YRRFRDPEPDNHYDF,I-Ab,15,14978.972656,59.159653,0.749,H2_Aa_b__H2_Ab_b,29.592826,28.847051
129020,ID8_CPR,MT,RRFRDPEPDNHYDFC,I-Ab,15,18422.242188,68.083969,4.560,H2_Aa_b__H2_Ab_b,32.111244,74.099816
129021,ID8_CPR,MT,RFRDPEPDNHYDFCL,I-Ab,15,22416.878906,76.982864,21.100,H2_Aa_b__H2_Ab_b,32.832248,83.528138
129022,ID8_CPR,MT,FRDPEPDNHYDFCLI,I-Ab,15,29518.529297,88.980057,69.400,H2_Aa_b__H2_Ab_b,33.348127,88.819515
